In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report,confusion_matrix,ConfusionMatrixDisplay,accuracy_score
import warnings
warnings.filterwarnings('ignore')

In [4]:
!pip install tensorflow
!pip install keras
!pip install tensorflow-datasets

  Using cached tensorflow-2.21.0-cp312-cp312-manylinux_2_27_x86_64.whl.metadata (4.4 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-manylinux2010_x86_64.whl.metadata (5.2 kB)
  Using cached h5py-3.14.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (2.7 kB)
  Using cached wheel-0.46.3-py3-none-any.whl.metadata (2.4 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.6/572.6 MB 769.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 124.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 101.9 MB/s eta 0:00:00
  Attempting uninstall: h5py
    Found existing installation: h5py 3.16.0
    Uninstalling h5py-3.16.0:
      Successfully

In [5]:
from keras.models import Sequential
from keras.layers import Dense,Conv2D,MaxPooling2D,Flatten,Dropout,BatchNormalization
from keras.callbacks import EarlyStopping


In [6]:
import tensorflow_datasets as tfds
import tensorflow as tf


In [7]:
import tensorflow as tf
import tensorflow_datasets as tfds

ds, info = tfds.load(
    'cats_vs_dogs',
    split='train',
    as_supervised=True,
    with_info=True
)

dataset_size = info.splits['train'].num_examples
print(f'Total dataset size: {dataset_size}')

ds = ds.shuffle(10000, reshuffle_each_iteration=False)

train_size = int(0.8 * dataset_size)

train_ds = ds.take(train_size)
test_ds = ds.skip(train_size)

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/1 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/cats_vs_dogs/incomplete.5V9M76_4.0.1/cats_vs_dogs-train.tfrecord*...:   0%…

Dataset cats_vs_dogs downloaded and prepared to /root/tensorflow_datasets/cats_vs_dogs/4.0.1. Subsequent calls will reuse this data.
Total dataset size: 23262


In [8]:
dataset_size = info.splits['train'].num_examples
print(dataset_size)  # ~23262

23262


In [9]:
ds = ds.shuffle(10000, reshuffle_each_iteration=False)

train_size = int(0.8 * dataset_size)

train_ds = ds.take(train_size)
test_ds = ds.skip(train_size)

In [10]:
IMG_SIZE = 128

def preprocess(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = image / 255.0
    return image, label

In [11]:
# Preprocess and batch the datasets
train_ds = train_ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
test_ds = test_ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)

BATCH_SIZE = 64

train_ds = train_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [12]:
# This cell is now empty as its content has been moved to ouOBq5WK7eAM

In [13]:
model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(128,128,3)),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(128, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(128, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.5),   # prevents overfitting

    tf.keras.layers.Dense(1, activation='sigmoid')
])

In [14]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [15]:
history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=10
)

Epoch 1/10
291/291 ━━━━━━━━━━━━━━━━━━━━ 233s 778ms/step - accuracy: 0.6135 - loss: 0.6430 - val_accuracy: 0.7217 - val_loss: 0.5357
Epoch 2/10
291/291 ━━━━━━━━━━━━━━━━━━━━ 241s 809ms/step - accuracy: 0.7461 - loss: 0.5205 - val_accuracy: 0.7780 - val_loss: 0.4754
Epoch 3/10
291/291 ━━━━━━━━━━━━━━━━━━━━ 238s 797ms/step - accuracy: 0.8047 - loss: 0.4245 - val_accuracy: 0.8025 - val_loss: 0.4349
Epoch 4/10
291/291 ━━━━━━━━━━━━━━━━━━━━ 238s 797ms/step - accuracy: 0.8484 - loss: 0.3475 - val_accuracy: 0.8435 - val_loss: 0.3620
Epoch 5/10
291/291 ━━━━━━━━━━━━━━━━━━━━ 235s 788ms/step - accuracy: 0.8746 - loss: 0.2966 - val_accuracy: 0.8517 - val_loss: 0.3467
Epoch 6/10
291/291 ━━━━━━━━━━━━━━━━━━━━ 234s 786ms/step - accuracy: 0.8953 - loss: 0.2536 - val_accuracy: 0.8586 - val_loss: 0.3314
Epoch 7/10
291/291 ━━━━━━━━━━━━━━━━━━━━ 235s 789ms/step - accuracy: 0.9105 - loss: 0.2189 - val_accuracy: 0.8521 - val_loss: 0.3541
Epoch 8/10
291/291 ━━━━━━━━━━━━━━━━━━━━ 232s 776ms/step - accuracy: 0.9258 -

In [18]:
loss,acc=model.evaluate(test_ds)
print(f"Accuracy: {acc:.4f}")

73/73 ━━━━━━━━━━━━━━━━━━━━ 22s 202ms/step - accuracy: 0.8502 - loss: 0.5439
Accuracy: 0.8502


In [17]:
loss

0.8502041697502136